In [1]:
import argparse
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
from model.dataset import BagsDataset, custom_collate_fn
from model.model import MIL, EarlyStopping
import scanpy as sc
import numpy as np

In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
#input_dir = '/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/VisiumHD/HumanLungCancer'
adata = sc.read_h5ad('/work/DPDS/s439765/data4spacer/spatial_transcriptomics/linghua_cosmx/GAME3D6/d6_sample1_Tcell.h5ad')
#output_dir = os.path.join('pretrained_models', input_dir.split('/')[-1])
output_dir='test'
os.makedirs(output_dir, exist_ok=True)

In [3]:
dataset = BagsDataset(adata, immune_cell='tcell', radius=50, n_genes=500, resolution='high')


Immune cell: T
[1 0 2]
Tumor cells shape after filtering: (30903, 20385)
Selecting top 500 genes based on mean expression


/work/OSPH/s439765/envs/spatial_tcr/lib/python3.9/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Preprocessed data: (56900, 585)


Creating Bags with radius 50: 100%|█████████████████████████| 56900/56900 [00:10<00:00, 5453.13it/s]


Total batches created: 2174


In [7]:
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

In [8]:
distances, gene_expressions, labels, core_index, gene_names, cell_ids = next(iter(train_loader))


In [9]:
labels

[tensor(1.), tensor(0.)]

In [10]:
class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)
    
    def forward(self, x):
        #print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x

In [11]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[2.7992e-40],
        [9.1376e-41],
        [7.0986e-41],
        [1.7600e-40],
        [0.0000e+00],
        [1.5665e-34],
        [6.9341e-32],
        [1.2216e-23],
        [1.2175e-16],
        [9.1453e-32],
        [7.3396e-12],
        [5.6052e-45],
        [1.5752e-31],
        [0.0000e+00],
        [1.0000e+00],
        [1.1210e-44],
        [7.0892e-42],
        [1.0340e-21],
        [1.5863e-15],
        [1.2129e-40],
        [1.7739e-25],
        [2.3274e-34],
        [1.1540e-41],
        [1.2190e-27],
        [8.2948e-27],
        [1.2742e-37],
        [3.9657e-43],
        [9.5749e-29],
        [2.0421e-37],
        [5.2174e-20]], grad_fn=<SoftmaxBackward0>)


In [12]:
class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x

In [13]:
gene_expressions[0]

tensor([[0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 3., 0., 1.],
        [0., 0., 0.,  ..., 1., 5., 1.],
        ...,
        [0., 0., 0.,  ..., 1., 3., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [14]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [2.5318e-37, 2.5318e-37, 2.5318e-37,  ..., 8.8111e-34, 2.5318e-37,
         3.8367e-36],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        ...,
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 0.0000e+00, 0.0000e+00,
         0.0000e+00]], grad_fn=<SoftmaxBackward0>)


In [15]:
class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes

In [16]:
all_genes = pd.read_csv('data/human_filtered.csv')
all_genes = all_genes['Gene'].values.tolist()

In [17]:
model = Immunogenicity(all_genes)

In [18]:
output, filtered_genes = model(list(gene_names[0]))
print(len(output))
print(filtered_genes)
df = pd.DataFrame({'Gene': filtered_genes})


177
['TAP2', 'IFI6', 'TOP2A', 'PBK', 'TPX2', 'PRAME', 'MUC1', 'MUC12', 'CEACAM1', 'EPCAM', 'PMEL', 'MLANA', 'LAGE3', 'HORMAD1', 'KRT8', 'KRT18', 'KRT19', 'ERBB2', 'MAGEA4', 'MAGEA10', 'AFP', 'CEACAM5', 'SOX2', 'SLC45A2', 'WT1', 'HOXB9', 'GUCY2C', 'CDK4', 'CDK6', 'AURKA', 'BCL2', 'BIRC5', 'CD274', 'FGL1', 'HK2', 'PARP1', 'RAD51', 'NANOG', 'EGFR', 'KRAS', 'OR5V1', 'OR2H1', 'TSPAN31', 'TMEM191C', 'IGSF8', 'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DOB', 'HLA-DQB2', 'HLA-DRB3', 'PGA4', 'PGC', 'LIPF', 'ATP4A', 'HLA-B', 'MUC6', 'GKN1', 'CLIC6', 'CC2D2A', 'KCNE2', 'MFSD4A', 'ATP4B', 'PGA3', 'PPP2R3C', 'KRT8', 'HLA-A', 'FXYD7', 'CCDC40', 'SLC9A3', 'TACR3', 'HLA-C', 'CLPS', 'B3GNT5', 'CPA2', 'TFF2', 'CCNYL1', 'PDXDC1', 'ATP7B', 'ALG8', 'SPANXD', 'GABRR1', 'GKN2', 'TRIM50', 'CLDN18', 'DNAJC4', 'POM121C', 'LRRC28', 'TRAF6', 'FGD4', 'NPIPB6', 'PABPN1', 'SPIN4', 'TCOF1', 'EIF4H', 'PDZRN4', 'SNRNP70', 'ZNF778', 'GOLGA8A', 'GAK', 'MUC1', 'MICALL1', 'RPS6KB1', 'NEK8', 'PRKCQ', 'SNURF', 'DDX39B', 'ARSD', 'FBRSL1'

In [19]:
class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv(soutput.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)

In [20]:
model = MIL(all_genes)

In [21]:
output = model(distances, gene_expressions, list(gene_names[0]))
output

tensor([0.7311, 0.8495], grad_fn=<SqueezeBackward1>)

In [22]:

labels[0]


tensor(1.)

In [23]:
ig_scores_before_training = model.immunogenicity.ig.clone().detach().cpu()

In [24]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch

In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:

model = MIL(all_genes).to(device)
criterion = nn.BCELoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 50
patience = 5
early_stopping = EarlyStopping(patience=patience, delta=0.00001)

In [28]:
current_genes = gene_names[0]

In [ ]:

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
        
    # Lists to store outputs and labels for AUROC calculation
    all_outputs = []
    all_labels = []
        
    with tqdm(train_loader, unit="batch") as tepoch:
        for i, batch_data in enumerate(tepoch):
            tepoch.set_description(f"Epoch {epoch+1}/{num_epochs}")
            optimizer.zero_grad()

            # Unpack the batch data
            distances_list, gene_expressions_list, labels_list, core_idxs_list, gene_names_list, cell_ids_list = batch_data
                
            # Move data to device and prepare labels
            distances_list = [distances.to(device) for distances in distances_list]
            gene_expressions_list = [gene_exp.to(device) for gene_exp in gene_expressions_list]
            labels = torch.stack(labels_list).float().to(device)
            current_genes_list = gene_names_list  # List of gene names for each bag

            # Forward pass
            outputs = model(distances_list, gene_expressions_list, current_genes_list)
                
            if outputs is None:
                 continue  # Skip this batch if the model returns None
                
            if outputs.shape[0] != labels.shape[0]:
                # Handle mismatch in batch sizes if necessary
                continue
                
            # Compute BCE loss
            if selection == 'negative':
                labels = 1 - labels
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
            running_loss += loss.item()
            tepoch.set_postfix(loss=loss.item())
                
            # Accumulate outputs and labels for AUROC calculation
            all_outputs.extend(outputs.detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    train_loss = running_loss / len(train_loader)
        
    # Compute Training AUROC
    try:
        epoch_auc = roc_auc_score(all_labels, all_outputs)
    except ValueError:
        epoch_auc = float('nan')  # Handle case where AUROC can't be computed
        
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {train_loss:.4f}, AUROC: {epoch_auc:.4f}')
        
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_all_outputs = []
    val_all_labels = []
    with torch.no_grad():
        with tqdm(val_loader, unit="batch") as vtepoch:
            for val_batch_data in vtepoch:
                # Unpack validation batch data
                val_distances_list, val_gene_expressions_list, val_labels_list, val_core_idxs_list, val_gene_names_list, val_cell_ids_list = val_batch_data
                    
                # Move data to device and prepare labels
                val_distances_list = [distances.to(device) for distances in val_distances_list]
                val_gene_expressions_list = [gene_exp.to(device) for gene_exp in val_gene_expressions_list]
                val_labels = torch.stack(val_labels_list).float().to(device)
                val_current_genes_list = val_gene_names_list  # List of gene names for each bag
                    
                # Forward pass
                val_outputs = model(val_distances_list, val_gene_expressions_list, val_current_genes_list)
                    
                if val_outputs is None:
                    continue  # Skip this batch if the model returns None
                    
                if val_outputs.shape[0] != val_labels.shape[0]:
                    # Handle mismatch in batch sizes if necessary
                    continue
                    
                # Compute BCE loss
                if selection == 'negative':
                    val_labels = 1 - val_labels
                loss = criterion(val_outputs, val_labels)
                val_loss += loss.item()
                vtepoch.set_postfix(val_loss=loss.item())
                    
                # Accumulate outputs and labels for AUROC calculation
                val_all_outputs.extend(val_outputs.detach().cpu().numpy())
                val_all_labels.extend(val_labels.cpu().numpy())
            
        val_loss /= len(val_loader)
            
            # Compute Validation AUROC
        try:
            val_epoch_auc = roc_auc_score(val_all_labels, val_all_outputs)
        except ValueError:
            val_epoch_auc = float('nan')  # Handle case where AUROC can't be computed
            
        print(f'Validation Loss: {val_loss:.4f}, Validation AUROC: {val_epoch_auc:.4f}')

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Best model saved with validation loss {val_loss:.4f}")
            
    torch.save(model.state_dict(), os.path.join(output_dir, f'model_epoch_{epoch+1}.pth'))
        
    a = model.distance.a.clone().detach().cpu().numpy()
    b = model.gene_expression.b.clone().detach().cpu()
    alpha = model.alpha.clone().detach().cpu()
    beta = model.beta.clone().detach().cpu()
    # Save metrics
    save_metrics(epoch+1, train_loss, val_loss, val_epoch_auc,a,b,alpha,beta, output_dir)

    # Save IG scores after each epoch
    spacer_scores_after_training = model.immunogenicity.ig.clone().detach().cpu()
    spacer_scores_after_training = [score.item() for score in spacer_scores_after_training]
    save_spacer_scores(epoch, all_genes, spacer_scores_before_training, spacer_scores_after_training, output_dir)

Epoch 1/50:   0%|          | 0/1956 [00:00<?, ?batch/s]


AttributeError: 'list' object has no attribute 'clone'

In [ ]:
import csv
ig_scores_after_training = model.immunogenicity.ig.clone().detach().cpu()
alpha = model.state_dict()['alpha'].item()
beta = model.state_dict()['beta'].item()
a = model.state_dict()['distance.a'].item()
b = model.state_dict()['gene_expression.b'].item()  
with open(os.path.join(output_dir, 'ig_score_changes.csv'), 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['alpha', alpha])
    writer.writerow(['beta', beta])
    writer.writerow(['a', a])
    writer.writerow(['b', b])
    writer.writerow(['Gene', 'IG Score Before Training', 'IG Score After Training'])
    for gene, before, after in zip(all_genes, ig_scores_before_training, ig_scores_after_training):
        writer.writerow([gene, before.item(), after.item()])
torch.save(model.state_dict(), os.path.join(output_dir, 'final_model.pth'))


In [34]:
output_dir

'pretrained_models/HumanLungCancer'

In [ ]:
model.state_dict()

OrderedDict([('alpha', tensor(0.4340)),
             ('beta', tensor(0.9409)),
             ('distance.a', tensor(-3.0000)),
             ('gene_expression.b', tensor(-0.9906)),
             ('immunogenicity.ig',
              tensor([-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]))])